In [118]:
import os
import pandas as pd
import ast
os.environ["TOGETHER_API_KEY"]="391ae92110a137c8da7c86fe6935c200fd5a620f146c6970c267c3f811eae0d5"

In [2]:
#!pip install together

In [119]:
from together import Together
client=Together()
model="meta-llama/Llama-3.3-70B-Instruct-Turbo"

In [120]:
# Reding test and train data
test = pd.read_csv('test_genai.csv')
train = pd.read_csv('train_genai.csv')

In [121]:
def get_response(prompt, model=model):
    messages = [{"role":"user","content":prompt}]
    client = Together()
    response = client.chat.completions.create(model=model,messages=messages)
    return response.choices[0].message.content

In [122]:
train.head()

,ticket_ID,ticket_subject,ticket_body,department,type,priority,language
0,1001,Discrepancia de facturación en Google Workspace,"Estimado equipo de soporte de TI,\n\nEstoy esc...",Billing and Payments,Incident,low,es
1,1002,Urgent Consultation Request for Critical IT Is...,"Dear IT Services Support Team, I hope this mes...",Customer Service,Request,high,en
2,1003,Consulta sobre Servicios de Consultoría en TI,"Estimado Servicio de Atención al Cliente,\n\nM...",General Inquiry,Request,medium,es
3,1004,Demande de mise à jour des dossiers,"Cher service client, \n\nJe vous écris pour de...",Human Resources,Change,low,fr
4,1005,Issues with Slack connection affecting team co...,"Dear Customer Support Team,\n\nI am encounteri...",Product Support,Problem,medium,en


In [124]:
output=[]
for value in train['ticket_body']:
  user_prompt = f''' classify the customer support ticket delimited by <>  in english such as
  department as one of Technical Support, Customer Service, Billing and Payments, Product Support, IT Support, Returns and Exchanges, Sales and Pre-Sales, Human Resources, Service Outages and Maintenance, or General Inquiry
  type as one of Incident, Request, Change, problem
  without any reasoning  in dictionary format<{value}>'''
  output.append(ast.literal_eval(get_response(user_prompt)))
  print(get_response(user_prompt))


SyntaxError: invalid syntax (<unknown>, line 1)

In [103]:
output = pd.DataFrame(output)
output.head()
train["department_pred"] = output.department
train["priority_pred"] = output.priority
train["language_pred"] = output.language
train["type_pred"] = output.type

In [104]:
train.head(10)

,ticket_ID,ticket_subject,ticket_body,department,type,priority,language,department_pred,priority_pred,language_pred,type_pred
0,1001,Discrepancia de facturación en Google Workspace,"Estimado equipo de soporte de TI,\n\nEstoy esc...",Billing and Payments,Incident,low,es,Billing and Payments,medium,Spanish,Incident
1,1002,Urgent Consultation Request for Critical IT Is...,"Dear IT Services Support Team, I hope this mes...",Customer Service,Request,high,en,IT Support,high,English,Incident
2,1003,Consulta sobre Servicios de Consultoría en TI,"Estimado Servicio de Atención al Cliente,\n\nM...",General Inquiry,Request,medium,es,Sales and Pre-Sales,low,Spanish,Request
3,1004,Demande de mise à jour des dossiers,"Cher service client, \n\nJe vous écris pour de...",Human Resources,Change,low,fr,IT Support,medium,French,Request
4,1005,Issues with Slack connection affecting team co...,"Dear Customer Support Team,\n\nI am encounteri...",Product Support,Problem,medium,en,Technical Support,medium,English,Incident
5,1006,Defective Dell XPS 13 9310,"Dear Tech Online Store Support,\n\nI received ...",Returns and Exchanges,Incident,low,en,Returns and Exchanges,high,English,Incident
6,1007,Touchscreen and Keyboard Issues with Surface P...,"Dear Tech Online Store Customer Support,\n\nI ...",Sales and Pre-Sales,Problem,medium,en,Technical Support,high,English,Incident
7,1008,AWS-Serverausfall,"Sehr geehrte Kundenbetreuung,\n\nwir hatten ei...",Service Outages and Maintenance,Incident,high,de,Technical Support,high,German,Incident
8,1009,Solicitud urgente de orientación y solución de...,"Estimado soporte al cliente, espero que este m...",Technical Support,Request,high,es,Technical Support,high,Spanish,Incident


In [85]:
# Specify the CSV file name
csv_file = "output.csv"

# Writing DataFrame to CSV
output.to_csv(csv_file, index=False)

print(f"DataFrame written to {csv_file}")

DataFrame written to output.csv
